In [ ]:
# 03 - Modelling and Identifying Undervalued Players

This notebook builds a Random Forest regression model to predict football player market value from performance statistics.

The aim is to compare predicted market value against actual market value and identify players who may be undervalued relative to their statistical output.

Key modelling safeguards:
- Value-related columns are removed from model features to prevent data leakage.
- Market value is log-transformed to reduce skew.
- Players with market value below €1m are excluded from undervaluation ranking to reduce noisy low-value outliers.

In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
df = pd.read_csv("../data/cleaned_players_dataset.csv")

print(df.shape)
df.head()

(1761, 147)


,Rk,Player,Nation,Pos,Squad,Age,Born,MP,Starts,Min,...,goal_contribution,goal_contribution_per90,shot_accuracy,attacking_score,position_group,Comp_de Bundesliga,Comp_eng Premier League,Comp_es La Liga,Comp_fr Ligue 1,Comp_it Serie A
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,25.0,2000.0,30,24,1975,...,7,0.319635,0.365854,0.506849,MID,False,True,False,False,False
1,3,Jones El-Abdellaoui,ma MAR,"MF,FW",Celta Vigo,20.0,2006.0,20,4,638,...,2,0.281690,0.250000,0.591549,MID,False,False,True,False,False
2,5,Himad Abdelli,dz ALG,MF,Angers,26.0,1999.0,13,11,943,...,2,0.190476,0.454545,0.304762,MID,False,False,False,True,False
3,6,Ali Abdi,tn TUN,"DF,MF",Nice,32.0,1993.0,17,11,955,...,1,0.094340,0.200000,0.141509,DEF,False,False,False,True,False
4,7,Salis Abdul Samed,gh GHA,MF,Nice,26.0,2000.0,14,9,663,...,0,0.000000,NaN,0.000000,MID,False,False,False,True,False


In [13]:
# Use the original dataset's market value field as the target.
# External valuation data was found to be inconsistent for this project.
target = "market_value_in_eur_x"


In [14]:
# Numeric features only
X = df.select_dtypes(include=["number"])

# Remove ALL value-related columns
value_cols = [col for col in X.columns if "value" in col.lower()]
X = X.drop(columns=value_cols, errors="ignore")

# Target (log transform)
y = np.log1p(df[target])

print("Removed columns:", value_cols)
print("X shape:", X.shape)

Removed columns: ['market_value_in_eur_x', 'highest_market_value_in_eur', 'market_value_in_eur_y']
X shape: (1761, 97)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [17]:
y_pred = model.predict(X_test)

In [21]:
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred)

mae = mean_absolute_error(y_test_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
r2 = r2_score(y_test_real, y_pred_real)

print(f"MAE: €{mae:,.0f}")
print(f"RMSE: €{rmse:,.0f}")
print(f"R²: {r2:.3f}")

MAE: €7,125,508
RMSE: €12,407,481
R²: 0.428


In [23]:
df["predicted_value"] = np.expm1(model.predict(X))
df["actual_value"] = df[target]

In [25]:
df["undervaluation_gap"] = df["predicted_value"] - df["actual_value"]

df["undervaluation_pct"] = (
    df["undervaluation_gap"] / df["actual_value"]
)

In [27]:
df_filtered = df[df["actual_value"] >= 1_000_000].copy()

print("Before filter:", len(df))
print("After filter:", len(df_filtered))

Before filter: 1761
After filter: 1645


In [29]:
## Identifying Undervalued Players

The model identifies players whose predicted market value significantly exceeds their actual market value. These players may represent potential undervalued opportunities.

However, it is important to note that the model tends to overestimate certain players, particularly attacking players with strong per-90 metrics. Therefore, these results should be interpreted as relative indicators rather than absolute valuations.

SyntaxError: invalid syntax (2055347528.py, line 3)

In [30]:
top_undervalued_gap = df_filtered.sort_values(
    "undervaluation_gap", ascending=False
).head(20)

top_undervalued_gap[[
    "Player",
    "actual_value",
    "predicted_value",
    "undervaluation_gap"
]]

,Player,actual_value,predicted_value,undervaluation_gap
450,Anastasios Douvikas,18000000,4.304383e+07,2.504383e+07
1401,Julian Ryerson,25000000,4.891609e+07,2.391609e+07
1532,Josip Stanišić,35000000,5.872021e+07,2.372021e+07
1609,Ferrán Torres,50000000,7.349549e+07,2.349549e+07
462,Aron Dønnum,5000000,2.472195e+07,1.972195e+07
1740,Nicola Zalewski,20000000,3.855506e+07,1.855506e+07
415,Federico Dimarco,50000000,6.852335e+07,1.852335e+07
874,Konrad Laimer,32000000,5.007644e+07,1.807644e+07
589,Yann Gboho,15000000,3.109503e+07,1.609503e+07
331,Félix Correia,10000000,2.577248e+07,1.577248e+07


In [31]:
top_undervalued_pct = df_filtered.sort_values(
    "undervaluation_pct", ascending=False
).head(20)

top_undervalued_pct[[
    "Player",
    "actual_value",
    "predicted_value",
    "undervaluation_gap",
    "undervaluation_pct"
]]

,Player,actual_value,predicted_value,undervaluation_gap,undervaluation_pct
522,Roberto Férnandez,1500000,1.058718e+07,9.087176e+06,6.058117
808,Dermane Karim,3000000,1.645141e+07,1.345141e+07,4.483803
462,Aron Dønnum,5000000,2.472195e+07,1.972195e+07,3.944391
787,Mateo Joseph,3500000,1.523094e+07,1.173094e+07,3.351698
1137,Joel Mvuka,2000000,8.527110e+06,6.527110e+06,3.263555
1696,Mërgim Vojvoda,3000000,1.153037e+07,8.530365e+06,2.843455
526,Jonathan Fischer,3000000,1.063000e+07,7.630002e+06,2.543334
1013,Toni Martínez,4000000,1.397123e+07,9.971232e+06,2.492808
1237,Victor Parada,2000000,6.910249e+06,4.910249e+06,2.455124
354,Matheus Cunha,2000000,6.333481e+06,4.333481e+06,2.166741


In [32]:
df.to_csv("../data/modelled_players_all.csv", index=False)
df_filtered.to_csv("../data/modelled_players_filtered.csv", index=False)
top_undervalued_gap.to_csv("../data/top_undervalued_players_gap.csv", index=False)
top_undervalued_pct.to_csv("../data/top_undervalued_players_pct.csv", index=False)